# Dynasty Title Table 文件的处理

historical object schema
- id: uuid
- type:
- name:
- parents: option, [id]
- start_time:
- end_time: option,
- tags

## 1. 朝代处理

In [19]:
%pip install --upgrade numpy pandas openpyxl

87106.24s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


    PyYAML (>=5.1.*)
            ~~~~~~^
Note: you may need to restart the kernel to use updated packages.


In [20]:
# helpers/time_parser.py
import re

def parse_time_string(time_str):
    """
    解析中国古代完整时间字符串
    1. 闰月 → 月份数字为负数 例：闰四月=-4、闰正月=-1
    2. 无年份/月份/日期 → 全部返回 None
    3. 公元前年份=负数，公元后=正数，模糊值(???年)=None
    4. 兼容：初一/初十/二十/廿一/廿九/卅日 全中文日期
    :param time_str: 待解析时间字符串 (支持 916年十二月-922年正月 这类区间格式)
    :return: dict{year:int/None, month:int/None, day:int/None}
    """
    year = None
    month = None
    day = None
    
    # 空值/无意义内容 直接返回全None
    if not time_str or str(time_str).strip() in ["", "???", "未知", "无"]:
        return {"year": year, "month": month, "day": day}
    
    time_str = str(time_str).strip()
    # ===================== 1. 年份解析 =====================
    bc_pattern = r"前(\d+)年"
    ad_pattern = r"(?<!前)(\d+)年"
    bc_match = re.search(bc_pattern, time_str)
    if bc_match:
        year = -int(bc_match.group(1))
    else:
        ad_match = re.search(ad_pattern, time_str)
        if ad_match:
            year = int(ad_match.group(1))

    # ===================== 2. 月份解析【优先级最高，核心修复关键】 =====================
    month_map = {
        '正月': 1, '一月': 1,
        '二月': 2, '三月': 3, '四月': 4, '五月': 5, '六月': 6,
        '七月': 7, '八月': 8, '九月': 9, '十月': 10,
        '十一月': 11, '冬月': 11,
        '十二月': 12, '腊月': 12
    }
    # 正则匹配顺序：先闰月 → 再长月份(十一月/十二月) → 再短月份，彻底避免误匹配
    leap_month_pattern = r"闰(正月|一月|二月|三月|四月|五月|六月|七月|八月|九月|十月|十一月|十二月|冬月|腊月)"
    long_month_pattern = r"(十一月|十二月|冬月|腊月)"
    short_month_pattern = r"(正月|一月|二月|三月|四月|五月|六月|七月|八月|九月|十月)"
    
    leap_month_match = re.search(leap_month_pattern, time_str)
    long_month_match = re.search(long_month_pattern, time_str)
    normal_month_match = re.search(short_month_pattern, time_str)
    
    if leap_month_match:
        month_cn = leap_month_match.group(1)
        month = -month_map[month_cn]  # 闰月=负数
    elif long_month_match:
        month_cn = long_month_match.group(1)
        month = month_map[month_cn]
    elif normal_month_match:
        month_cn = normal_month_match.group(1)
        month = month_map[month_cn]

    # ===================== 3. 日期解析 =====================
    num_map = {'初':0, '十':10, '廿':20, '卅':30,
               '一':1, '二':2, '三':3, '四':4, '五':5, '六':6, '七':7, '八':8, '九':9}
    # 只匹配：正月初一、三月二十、闰四月廿九、腊月卅日 这类合法日期格式
    day_pattern = r"(正月|一月|二月|三月|四月|五月|六月|七月|八月|九月|十月|十一月|十二月|冬月|腊月|闰.+?月)(初|十|廿|卅)[一二三四五六七八九]{0,1}日?"
    day_match = re.search(day_pattern, time_str)
    
    if day_match:
        # 只截取日期部分，剔除前面的月份
        day_cn = day_match.group().split(day_match.group(1))[-1].replace('日','').strip()
        day = 0
        for char in day_cn:
            day += num_map.get(char, 0)
        day = day if day != 0 else 1  # 初一 → 1

    return {"year": year, "month": month, "day": day}

In [21]:
# helpers/time_parser.py
def parse_qiqi_time(qiqi_time_str):
    """
    # 处理中国古代起讫时间字符串，提取起始和终止时间
    # 前1042年-前1021年
    # 前???年-前???年
    # 916年十二月-922年正月
    # 983年六月-1021年闰四月
    # 1086年正月-七月
    # 1129年三月十一日-四月初三
    :param qiqi_time_str: 待解析时间字符串 (支持 916年十二月-922年正月 这类区间格式)
    :return: dict{year:int/None, month:int/None, day:int/None}, dict{year:int/None, month:int/None, day:int/None}
    """
    if not isinstance(qiqi_time_str, str):
        return None, None
    
    parts = qiqi_time_str.split('-')
    if len(parts) != 2:
        return None, None
    
    start_time = parts[0].strip()
    end_time = parts[1].strip()

    start_time = parse_time_string(start_time)
    end_time = parse_time_string(end_time)
    if (end_time['year'] == None):
        end_time['year'] = start_time['year']

    return start_time, end_time

In [22]:
# 测试代码
qiqi_time_strs = [
  '前1042年-前1021年',
  '前???年-前???年',
  '916年十二月-922年正月',
  '983年六月-1021年闰四月',
  '1086年正月-七月',
  '1129年三月十一日-四月初三'
]

for qiqi_time_str in qiqi_time_strs:
    start_time, end_time = parse_qiqi_time(qiqi_time_str)
    print(f"时间字符串：{qiqi_time_str} → 起始时间：{start_time}，终止时间：{end_time}")

时间字符串：前1042年-前1021年 → 起始时间：{'year': -1042, 'month': None, 'day': None}，终止时间：{'year': -1021, 'month': None, 'day': None}
时间字符串：前???年-前???年 → 起始时间：{'year': None, 'month': None, 'day': None}，终止时间：{'year': None, 'month': None, 'day': None}
时间字符串：916年十二月-922年正月 → 起始时间：{'year': 916, 'month': 12, 'day': None}，终止时间：{'year': 922, 'month': 1, 'day': None}
时间字符串：983年六月-1021年闰四月 → 起始时间：{'year': 983, 'month': 6, 'day': None}，终止时间：{'year': 1021, 'month': -4, 'day': None}
时间字符串：1086年正月-七月 → 起始时间：{'year': 1086, 'month': 1, 'day': None}，终止时间：{'year': 1086, 'month': 7, 'day': None}
时间字符串：1129年三月十一日-四月初三 → 起始时间：{'year': 1129, 'month': 3, 'day': 11}，终止时间：{'year': 1129, 'month': 4, 'day': 3}


In [23]:
import pandas as pd
import json

# 列索引常量定义
USE_COL = 1        # 是否使用
PERIOD_COL = 2     # 历史分期（分期）
SUB_PERIOD_COL = 3 # 历史分期（时代）
CATEGORY_COL = 4   # 分类
DYNASTY_COL = 5    # 主朝代
POLITY_COL = 6     # 政权
TIME_COL = 13      # 起讫时间

In [32]:
# 从Excel文件中提取历史分期数据的ETL函数
def etl_historical_periods(excel_path, sheet_name, period_type):
    # 1. 读取xlsx文件，指定openpyxl引擎解析xlsx格式
    df = pd.read_excel(
        io=excel_path,
        sheet_name=sheet_name,
        engine="openpyxl",
        dtype=str,                  # 强制所有单元格按字符串读取，避免类型错误
        header=None,                # 【关键1】不把第一行当表头，强制读取所有行（含空行）
        skiprows=3,                 # 【关键2】不跳过任何行
        usecols=None,               # 【关键3】不跳过任何列
        nrows=None,                 # 【关键4】读取全部行，不限行数
        na_filter=False             # 【关键5】不自动过滤空值，保留所有单元格内容
    )

    # 2. 筛选需要的列 + 删除空行，只保留有【大时代】和【子时代】的有效数据
    df_data = df[[period_type, TIME_COL]].copy()
    df_data = df_data[
        (df_data[period_type].str.strip()!="")
    ]
    # 重置索引，防止遍历的时候索引错乱
    df_data = df_data.reset_index(drop=True)

    print(f"数据行数：{len(df_data)}")
    print(df_data.head(5))

    # 3. 定义结果列表，按规则提取数据
    result_list = []
    total_rows = len(df_data)

    # 遍历每一行数据
    current_period = None
    for idx in range(total_rows):
        current_row = df_data.iloc[idx]
        period = current_row[period_type]
        qiqi_time = current_row[TIME_COL]

        start_time, _ = parse_qiqi_time(str(qiqi_time).strip())
        if (
            len(str(period).strip()) > 0 and 
            len(result_list) > 0
        ):
                result_list[len(result_list) - 1]["end_time"] = start_time
        
        # 遇到【中华民国】立即终止循环，且不加入结果
        if str(period).strip() == "中华民国":
            break

        # 追加到结果列表
        result_list.append({
            "type": "HISTORICAL_PERIOD",
            "name": str(period).strip(),
            "start_time": start_time,
            "end_time": None,
            "category": 'era_period' if period_type == PERIOD_COL else 'dynasty_period',
            "category:region": 'China',
        })

    return result_list

In [25]:
# 从Excel文件中提取历史分期数据的ETL函数
def etl_historical_periods0(excel_path, sheet_name):
    # 1. 读取xlsx文件，指定openpyxl引擎解析xlsx格式
    df = pd.read_excel(
        io=excel_path,
        sheet_name=sheet_name,
        engine="openpyxl",
        dtype=str,                  # 强制所有单元格按字符串读取，避免类型错误
        header=None,                # 【关键1】不把第一行当表头，强制读取所有行（含空行）
        skiprows=3,                 # 【关键2】不跳过任何行
        usecols=None,               # 【关键3】不跳过任何列
        nrows=None,                 # 【关键4】读取全部行，不限行数
        na_filter=False             # 【关键5】不自动过滤空值，保留所有单元格内容
    )

    # 2. 筛选需要的列 + 删除空行，只保留有【大时代】和【子时代】的有效数据
    df_data = df[[PERIOD_COL, SUB_PERIOD_COL, CATEGORY_COL, DYNASTY_COL, POLITY_COL, TIME_COL]].copy()
    df_data = df_data[
        (df_data[PERIOD_COL].str.strip()!="") |
        (df_data[SUB_PERIOD_COL].str.strip()!="") |
        (df_data[CATEGORY_COL].str.strip()!="") |
        (df_data[DYNASTY_COL].str.strip()!="") 
    ]
    # 重置索引，防止遍历的时候索引错乱
    df_data = df_data.reset_index(drop=True)

    print(f"数据行数：{len(df_data)}")
    print(df_data.head(5))

    # 3. 定义结果列表，按规则提取数据
    dynasty_list = []
    total_rows = len(df_data)

    # 遍历每一行数据
    current_period = None
    current_sub_period = None
    current_category = None
    for idx in range(total_rows):
        current_row = df_data.iloc[idx]
        period = current_row[PERIOD_COL]
        sub_period = current_row[SUB_PERIOD_COL]
        category = current_row[CATEGORY_COL]
        dynasty = current_row[DYNASTY_COL]
        qiqi_time = current_row[TIME_COL]

        start_time, end_time = parse_qiqi_time(str(qiqi_time).strip())
        if (len(str(dynasty).strip()) > 0 and len(dynasty_list) > 0):
                dynasty_list[len(dynasty_list) - 1]["end_time"] = end_time
        
        if (len(str(period).strip()) > 0):
                current_period = period
        if (len(str(sub_period).strip()) > 0):
                current_sub_period = sub_period
        if (len(str(category).strip()) > 0):
                current_category = category

        # 遇到【中华民国】立即终止循环，且不加入结果
        if str(period).strip() == "中华民国":
            break

        # 追加到结果列表
        dynasty_list.append({
            "type": "DYNASTY",
            "name": str(dynasty).strip(),
            "start_time": start_time,
            "end_time": None,
            "category:period": current_period,
            "category:sub_period": current_sub_period,
            "category": current_category
        })

    return dynasty_list

In [26]:
# 转为标准JSON格式（ensure_ascii=False保证中文正常显示，indent美化格式）
def dump_to_json(data, output_path):
    final_json = json.dumps(data, ensure_ascii=False, indent=4)

    # 打印JSON结果
    print("✅ 提取完成，最终JSON结果：")
    print(final_json)

    if (output_path):
        # 可选：将JSON结果保存到本地文件（推荐，方便查看）
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(final_json)
        print(f"\n✅ JSON文件已保存至：{output_path}")

In [33]:
excel_path = '../../docs/Dynasty Title Table20231219.xlsx'
sheet_name = '中国历代年号考'

era_periods = etl_historical_periods(excel_path, sheet_name, PERIOD_COL)
dynasty_periods = etl_historical_periods(excel_path, sheet_name, SUB_PERIOD_COL)
#historical_period_path = '../../library/meta/historical_periods.json'
historical_period_path = '../components/timelinechart/test/data/historical_periods.json'
dump_to_json(era_periods + dynasty_periods, historical_period_path)

数据行数：22
        2               13
0       先秦   前1046年-前1043年
1       秦汉     前221年-前210年
2  三国两晋南北朝     220年十月-226年
3   隋唐五代十国     581年二月-600年
4     宋辽金元  960年正月-963年十一月
数据行数：19
   3              13
0  西周  前1046年-前1043年
1  春秋    前770年-前720年
2  战国    前476年-前469年
3   秦    前221年-前210年
4  西汉    前202年-前195年
✅ 提取完成，最终JSON结果：
[
    {
        "type": "HISTORICAL_PERIOD",
        "name": "先秦",
        "start_time": {
            "year": -1046,
            "month": null,
            "day": null
        },
        "end_time": {
            "year": -221,
            "month": null,
            "day": null
        },
        "category": "era_period",
        "category:region": "China"
    },
    {
        "type": "HISTORICAL_PERIOD",
        "name": "秦汉",
        "start_time": {
            "year": -221,
            "month": null,
            "day": null
        },
        "end_time": {
            "year": 220,
            "month": 10,
            "day": null
        },
        "category": "era_peri